In [20]:
df = "/Users/alessiadelconte/codici/progetto NPL/DATASET/merged_hotels.json"

In [21]:
import json
import pandas as pd
import numpy as np

path = "/Users/alessiadelconte/codici/progetto NPL/DATASET/merged_hotels.json"

# Carica i dati grezzi
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Trasforma in una lista di righe (una per ogni recensione)
rows = []
for hotel in data:
    # Estraiamo i dati dell'hotel che ci servono come contesto per l'NLP
    hotel_meta = {
        'hotel_name': hotel.get('name'),
        'city': hotel.get('city'),
        'hotel_rating': hotel.get('rating'),
        'price': hotel.get('price')
    }
    
    for rev in hotel.get('reviews', []):
        if isinstance(rev, dict): # Sicurezza contro dati sporchi
            row = hotel_meta.copy()
            row.update({
                'comment': rev.get('comment', ''),
                'score': rev.get('score'),
                'date': rev.get('date')
            })
            rows.append(row)

# Ora creiamo il vero DataFrame
df = pd.DataFrame(rows)

In [22]:
# errors='coerce' trasforma i "N/A" in NaN (valori nulli gestibili)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['score'] = pd.to_numeric(df['score'], errors='coerce')

# Rimuovi eventuali recensioni senza testo (inutili per NLP)
df = df.dropna(subset=['comment'])

In [23]:
# Calcola la lunghezza dei commenti
df['word_count'] = df['comment'].str.split().str.len()

print(f"Totale recensioni caricate: {len(df)}")
print(f"Lunghezza media commenti: {df['word_count'].mean():.2f} parole")

Totale recensioni caricate: 3881
Lunghezza media commenti: 17.15 parole


In [24]:
import nltk
import os
print(nltk.data.path)
# try loading
try:
    from nltk.corpus import stopwords
    sw = stopwords.words('spanish')
    print('Spanish stopwords OK, count:', len(sw))
except Exception as e:
    print('Not available:', e)

['/Users/alessiadelconte/nltk_data', '/Library/Frameworks/Python.framework/Versions/3.12/nltk_data', '/Library/Frameworks/Python.framework/Versions/3.12/share/nltk_data', '/Library/Frameworks/Python.framework/Versions/3.12/lib/nltk_data', '/usr/share/nltk_data', '/usr/local/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/lib/nltk_data']
Not available: 
**********************************************************************
  Resource 'stopwords' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('stopwords')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'corpora/stopwords'

  Searched in:
    - '/Users/alessiadelconte/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/share/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share

In [25]:
import json, re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg') # Necessario per salvare i file senza visualizzarli a schermo
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# ─── CONFIGURAZIONE PERCORSI (ADATTATI AL TUO MAC) ──────────────────────────
BASE_PATH = "/Users/alessiadelconte/codici/progetto NPL"
INPUT_FILE = os.path.join(BASE_PATH, "DATASET/merged_hotels.json")
OUTPUT_DIR = os.path.join(BASE_PATH, "output_eda_completa")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── STOPWORDS MANUALI ──────────────────────────────────────────────────────
STOP_ES = set("""
a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante
e el ella ellas ello ellos en entre era eras éramos eran es esa esas ese eso esos esta estas
este esto estos fue fueron fui había hacía han has hasta he hemos hube hubo la las le les lo
los me mi mis muy más no nos nuestro nuestros o os otro otros para pero por porque que quien
se si sin sobre son su sus también te tenía todo todos tu tus un una unas unos ya yo
muy bien bueno buena buenos buenas es muy ha sido hotel habitación habitaciones todo todos
toda todas nada poco nada está están aquí allí tan così como que quale quali ogni hay ser estar tener
al del 10 de 10 8 de 8 9 de 9 7 de 7 excepcional muy bueno notable excelente
""".split())
STOP_EN = set("""
the a an and or but in on at to for of with is was are were be been having
have has had do does did will would could should may might shall can
this that these those it its i you he she we they me him her us them
my your his her our their mine yours his hers ours theirs
all any both each few more most other some such no nor only own same than then
very just as also so if about after before between because through during
""".split())
STOPWORDS = STOP_ES | STOP_EN

# ─── CARICAMENTO DATI SICURO ────────────────────────────────────────────────
try:
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ File caricato: {INPUT_FILE}")
except FileNotFoundError:
    print(f"❌ Errore: File non trovato in {INPUT_FILE}")
    exit()

rows = []
for hotel in data:
    if not isinstance(hotel, dict): continue
    
    # Pulizia sicura di Prezzo e Rating
    try:
        price = float(str(hotel.get('price', 0)).replace(',', '').strip())
    except:
        price = np.nan
    
    rating_hotel = pd.to_numeric(hotel.get('rating'), errors='coerce')
    
    for rev in hotel.get('reviews', []):
        if isinstance(rev, dict): # Prevenzione errore "string indices"
            rows.append({
                'hotel_name': hotel.get('name', 'N/A'),
                'city': hotel.get('city', 'N/A'),
                'hotel_rating': rating_hotel,
                'hotel_price': price,
                'n_services': len(hotel.get('services', [])),
                'comment': rev.get('comment', ''),
                'date': rev.get('date', ''),
                'score': pd.to_numeric(rev.get('score'), errors='coerce'),
                'source': rev.get('source', ''),
            })

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df['comment_len'] = df['comment'].str.len()
df['word_count'] = df['comment'].str.split().str.len()
df['score_bin'] = pd.cut(df['score'], bins=[0,6,8,10], labels=['Basso (≤6)','Medio (7-8)','Alto (9-10)'])

✅ File caricato: /Users/alessiadelconte/codici/progetto NPL/DATASET/merged_hotels.json


In [26]:
from langdetect import detect

def safe_detect(text):
    try:
        if len(str(text)) > 10:
            return detect(text)
        return "unknown"
    except:
        return "error"

# Procedi con l'applicazione
df['lang'] = df['comment'].apply(safe_detect)

In [27]:
# Calcolo delle frequenze
lang_counts = df['lang'].value_counts()
lang_pct = df['lang'].value_counts(normalize=True) * 100

print("\n" + "="*30)
print("   REPORT DISTRIBUZIONE LINGUE")
print("="*30)

for lang, count in lang_counts.items():
    # Usiamo .upper() per rendere il codice lingua (es: ES, EN) più leggibile
    print(f"{lang.upper():<10} : {count:>5} recensioni ({lang_pct[lang]:>5.1f}%)")

print("-"*30)
print(f"TOTALE     : {len(df):>5} recensioni")
print("="*30)

# Un controllo rapido per la tua tesi/progetto
n_en = lang_counts.get('en', 0)
if n_en > 0:
    print(f"\n💡 Nota: Hai {n_en} recensioni in inglese.")
    print("Assicurati che le STOPWORDS includano 'was', 'were', 'staff', 'room'.")


   REPORT DISTRIBUZIONE LINGUE
ES         :  3410 recensioni ( 87.9%)
EN         :   174 recensioni (  4.5%)
CA         :    93 recensioni (  2.4%)
PT         :    60 recensioni (  1.5%)
FR         :    42 recensioni (  1.1%)
SV         :    15 recensioni (  0.4%)
DE         :    14 recensioni (  0.4%)
IT         :    14 recensioni (  0.4%)
RO         :    12 recensioni (  0.3%)
KO         :    10 recensioni (  0.3%)
UNKNOWN    :     6 recensioni (  0.2%)
FI         :     4 recensioni (  0.1%)
NL         :     4 recensioni (  0.1%)
AF         :     3 recensioni (  0.1%)
TL         :     3 recensioni (  0.1%)
SL         :     3 recensioni (  0.1%)
ID         :     3 recensioni (  0.1%)
VI         :     2 recensioni (  0.1%)
DA         :     2 recensioni (  0.1%)
SW         :     2 recensioni (  0.1%)
SO         :     2 recensioni (  0.1%)
HR         :     1 recensioni (  0.0%)
ET         :     1 recensioni (  0.0%)
LT         :     1 recensioni (  0.0%)
------------------------------
T

In [28]:
# Creiamo un nuovo DataFrame "pulito" solo con recensioni in spagnolo
df_es = df[df['lang'] == 'es'].copy()

print(f"Dataset originale: {len(df)} righe")
print(f"Dataset filtrato (solo ES): {len(df_es)} righe")

# Ora usa df_es per rifare i grafici 07 e 08

Dataset originale: 3881 righe
Dataset filtrato (solo ES): 3410 righe


In [29]:


# ─── TEXT CLEANING ──────────────────────────────────────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+/\d+', '', text)
    text = re.sub(r'[^\w\sáéíóúüñà]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return tokens, ' '.join(tokens)

df[['tokens','clean_text']] = df['comment'].apply(lambda x: pd.Series(clean_text(x)))

# Colori per i grafici
CITY_COLORS = {'Barcelona':'#2196F3','Madrid':'#FF9800','Bilbao':'#4CAF50'}
SRC_COLORS  = {'booking':'#9C27B0','hotels_com':'#F44336','expedia':'#00BCD4'}

# ══════════════════════════════════════════════════════════════════════════
# FIG 1 — OVERVIEW (Salvataggio)
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
city_rev = df.groupby('city').size().sort_values(ascending=False)
axes[0].bar(city_rev.index, city_rev.values, color=[CITY_COLORS.get(c, '#607D8B') for c in city_rev.index])
axes[0].set_title('Recensioni per Città')

src_rev = df.groupby('source').size().sort_values(ascending=False)
axes[1].bar(src_rev.index, src_rev.values, color=[SRC_COLORS.get(s, '#607D8B') for s in src_rev.index])
axes[1].set_title('Per Piattaforma')

axes[2].hist(df.groupby('hotel_name').size(), bins=15, color='#607D8B')
axes[2].set_title('Distribuzione Recensioni/Hotel')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_overview.png'))
plt.close()
print("Fig 1 salvata ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 2 — DISTRIBUZIONE DEGLI SCORE
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Step 2 — Distribuzione degli Score', fontsize=16, fontweight='bold')

# 2a: Distribuzione globale
score_counts = df['score'].value_counts().sort_index()
axes[0].bar(score_counts.index, score_counts.values, color='#1976D2', edgecolor='white')
axes[0].set_title('Distribuzione Globale degli Score', fontweight='bold')
axes[0].axvline(df['score'].mean(), color='red', linestyle='--', label=f'Media: {df["score"].mean():.2f}')
axes[0].legend()

# 2b: Boxplot score per città
bp_data = [df[df['city']==c]['score'].dropna().values for c in ['Barcelona','Madrid','Bilbao']]
axes[1].boxplot(bp_data, labels=['Barcelona','Madrid','Bilbao'], patch_artist=True)
axes[1].set_title('Score per Città', fontweight='bold')

# 2c: Score medio per piattaforma
score_src = df.groupby('source')['score'].mean().sort_values()
axes[2].barh(score_src.index, score_src.values, color='#9C27B0')
axes[2].set_title('Score Medio per Piattaforma', fontweight='bold')
axes[2].set_xlim(min(score_src.values)-0.5, 10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_score_distribution.png'), dpi=150)
plt.close(); print("Fig 2 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 3 — ANALISI TEMPORALE
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Step 3 — Analisi Temporale', fontsize=16, fontweight='bold')

df_time = df.dropna(subset=['year'])
df_time = df_time[df_time['year'] >= 2019] # Filtro per anni recenti

# 3a: Volume per anno
for city, color in CITY_COLORS.items():
    yearly = df_time[df_time['city']==city].groupby('year').size()
    axes[0].plot(yearly.index, yearly.values, marker='o', label=city, color=color)
axes[0].set_title('Volume Recensioni per Anno', fontweight='bold')
axes[0].legend()

# 3b: Score medio nel tempo
score_year = df_time.groupby('year')['score'].mean()
axes[1].plot(score_year.index, score_year.values, marker='s', color='#E91E63', linewidth=2)
axes[1].set_title('Andamento Score Medio', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_temporal.png'), dpi=150)
plt.close(); print("Fig 3 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 4 — ANALISI DELLA LUNGHEZZA (PRE-NLP)
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Step 4 — Analisi della Lunghezza Testi', fontsize=16, fontweight='bold')

axes[0].hist(df['word_count'], bins=30, color='#3F51B5', edgecolor='white')
axes[0].set_title('Distribuzione Numero di Parole', fontweight='bold')

# Parole medie per fascia di score (Basso vs Alto)
wc_bins = df.groupby('score_bin', observed=True)['word_count'].mean()
axes[1].bar(wc_bins.index, wc_bins.values, color=['#F44336','#FF9800','#4CAF50'])
axes[1].set_title('Parole Medie per Fascia Score', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_length_analysis.png'))
plt.close(); print("Fig 4 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 5 — N-GRAMS (BIGRAMMI)
# ══════════════════════════════════════════════════════════════════════════
def get_top_ngrams(corpus, n=None, k=15):
    vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    return sorted(words_freq, key = lambda x: x[1], reverse=True)[:k]

top_bigrams = get_top_ngrams(df['clean_text'], n=2)
b_words, b_counts = zip(*top_bigrams)

plt.figure(figsize=(10, 6))
plt.barh(b_words, b_counts, color='#388E3C')
plt.title('Top 15 Bigrammi (Coppie di parole più frequenti)', fontweight='bold')
plt.gca().invert_yaxis()
plt.savefig(os.path.join(OUTPUT_DIR, '05_ngrams.png'), bbox_inches='tight')
plt.close(); print("Fig 5 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 6 — WORD CLOUDS PER CITTÀ
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, city in zip(axes, ['Barcelona','Madrid','Bilbao']):
    text = ' '.join(df[df['city']==city]['clean_text'])
    wc = WordCloud(width=400, height=400, background_color='white', max_words=50).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Top parole: {city}', fontsize=14, fontweight='bold')
    ax.axis('off')

plt.savefig(os.path.join(OUTPUT_DIR, '06_wordclouds.png'))
plt.close(); print("Fig 6 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 7 — TOPIC MODELING (LDA)
# ══════════════════════════════════════════════════════════════════════════
# 
vectorizer = CountVectorizer(max_df=0.9, min_df=5, max_features=500)
X = vectorizer.fit_transform(df['clean_text'])
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

words = vectorizer.get_feature_names_out()
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[-10:]
    axes[i].barh([words[j] for j in top_words_idx], [topic[j] for j in top_words_idx], color='teal')
    axes[i].set_title(f'Topic {i+1}')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_lda_topics.png'))
plt.close(); print("Fig 7 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 8 — TF-IDF CORRELAZIONE CON LO SCORE
# ══════════════════════════════════════════════════════════════════════════
# Vediamo quali parole sono associate a voti alti o bassi
tfidf = TfidfVectorizer(max_features=100)
X_tfidf = tfidf.fit_transform(df['clean_text'])
corrs = [np.corrcoef(X_tfidf[:,i].toarray().flatten(), df['score'].fillna(0))[0,1] for i in range(X_tfidf.shape[1])]

feat_names = tfidf.get_feature_names_out()
sorted_indices = np.argsort(corrs)

plt.figure(figsize=(10, 8))
plt.barh([feat_names[i] for i in sorted_indices[-15:]], [corrs[i] for i in sorted_indices[-15:]], color='green')
plt.barh([feat_names[i] for i in sorted_indices[:15]], [corrs[i] for i in sorted_indices[:15]], color='red')
plt.title('Parole più correlate con Score Alto (Verde) e Basso (Rosso)')
plt.savefig(os.path.join(OUTPUT_DIR, '08_tfidf_corr.png'), bbox_inches='tight')
plt.close(); print("Fig 8 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 9 — DISTRIBUZIONE DEI TOPIC PER CITTÀ
# ══════════════════════════════════════════════════════════════════════════
topic_values = lda.transform(X)
df['dominant_topic'] = topic_values.argmax(axis=1)
city_topics = pd.crosstab(df['city'], df['dominant_topic'], normalize='index')

city_topics.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis')
plt.title('Distribuzione dei Topic LDA per ogni Città')
plt.ylabel('Proporzione')
plt.legend(title='Topic', bbox_to_anchor=(1.05, 1))
plt.savefig(os.path.join(OUTPUT_DIR, '09_topic_by_city.png'), bbox_inches='tight')
plt.close(); print("Fig 9 ✓")

# Esempio rapido per WordCloud
plt.figure(figsize=(10, 5))
all_text = ' '.join(df['clean_text'])
wc = WordCloud(width=800, height=400, background_color='white').generate(all_text)
plt.imshow(wc)
plt.axis('off')
plt.savefig(os.path.join(OUTPUT_DIR, '06_wordcloud_generale.png'))
plt.close()

print(f"\n🚀 Analisi completata! Trovi tutti i grafici in:\n{OUTPUT_DIR}")

Fig 1 salvata ✓
Fig 2 ✓
Fig 3 ✓
Fig 4 ✓
Fig 5 ✓
Fig 6 ✓
Fig 7 ✓
Fig 8 ✓
Fig 9 ✓

🚀 Analisi completata! Trovi tutti i grafici in:
/Users/alessiadelconte/codici/progetto NPL/output_eda_completa


In [30]:
# ─── AGGIORNAMENTO STOPWORDS (DOMAIN SPECIFIC) ──────────────────────────────
# Queste parole compaiono in quasi tutte le recensioni e nascondono i veri insight
STOP_HOTEL = set("""
hotel habitacion habitaciones ubicacion ubicación personal staff estancia noche noches
dia dias días cliente clientes alojamiento establecimiento reserva check out
grazie grazie mille hola si no bien muy hotelero hostal hostales dormitorio
cama camas cuarto cuartos edificio planta piso estaba estaba estaba tiene tenía tenia hay había habia es era eran son 
está esta estan están fue fueron ser estar tener haber 
puedo puede possono ver bilbao madrid barcelona barcellona unos dos  
""".split())

# Uniamo le nuove stopwords alle precedenti
STOPWORDS = STOPWORDS | STOP_HOTEL

# ─── AGGIORNAMENTO TEXT CLEANING (PER SICUREZZA) ─────────────────────────────
def clean_text(text):
    text = str(text).lower()
    # Rimuoviamo i numeri e i pattern di score
    text = re.sub(r'\d+/\d+', '', text)
    # Teniamo solo lettere e spazi (gestendo i caratteri speciali spagnoli)
    text = re.sub(r'[^\w\sáéíóúüñà]', ' ', text)
    # Normalizzazione spazi
    text = re.sub(r'\s+', ' ', text).strip()
    # Filtriamo per stopwords e lunghezza minima (3 caratteri)
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return tokens, ' '.join(tokens)

# Riapplichiamo la pulizia al DataFrame
df[['tokens','clean_text']] = df['comment'].apply(lambda x: pd.Series(clean_text(x)))

print("✨ Testi ripuliti dalle Domain Stopwords!")

✨ Testi ripuliti dalle Domain Stopwords!


In [31]:
# ══════════════════════════════════════════════════════════════════════════
# FIG 7 — TOPIC MODELING (LDA)
# ══════════════════════════════════════════════════════════════════════════
# 
vectorizer = CountVectorizer(max_df=0.9, min_df=5, max_features=500)
X = vectorizer.fit_transform(df['clean_text'])
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

words = vectorizer.get_feature_names_out()
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[-10:]
    axes[i].barh([words[j] for j in top_words_idx], [topic[j] for j in top_words_idx], color='teal')
    axes[i].set_title(f'Topic {i+1}')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_lda_topics-post.png'))
plt.close(); print("Fig 7 ✓")

# ══════════════════════════════════════════════════════════════════════════
# FIG 8 — TF-IDF CORRELAZIONE CON LO SCORE
# ══════════════════════════════════════════════════════════════════════════
# Vediamo quali parole sono associate a voti alti o bassi
tfidf = TfidfVectorizer(max_features=100)
X_tfidf = tfidf.fit_transform(df['clean_text'])
corrs = [np.corrcoef(X_tfidf[:,i].toarray().flatten(), df['score'].fillna(0))[0,1] for i in range(X_tfidf.shape[1])]

feat_names = tfidf.get_feature_names_out()
sorted_indices = np.argsort(corrs)

plt.figure(figsize=(10, 8))
plt.barh([feat_names[i] for i in sorted_indices[-15:]], [corrs[i] for i in sorted_indices[-15:]], color='green')
plt.barh([feat_names[i] for i in sorted_indices[:15]], [corrs[i] for i in sorted_indices[:15]], color='red')
plt.title('Parole più correlate con Score Alto (Verde) e Basso (Rosso)')
# Cambia il nome del file per essere sicura di vedere quello nuovo
plt.savefig(os.path.join(OUTPUT_DIR, '08_tfidf_corr_FINALE.png'), bbox_inches='tight')
plt.close(); print("Fig 8 ✓")

Fig 7 ✓
Fig 8 ✓


In [32]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import os

# 1. LA "KILL LIST" DEFINITIVA
# Aggiungiamo 'great' e altre parole inutili che ho visto nel tuo ultimo grafico
PAROLE_DA_ELIMINARE = {'great', 'gracias', 'general', 'duda', 'además', 'mucho', 'agua', 'min', 'calle', 'ver', 'falta'}

# 2. PULIZIA SPIETATA
# Prendiamo il clean_text e strappiamo via quelle parole con la forza
df_es['clean_text_final'] = df_es['clean_text'].apply(
    lambda x: " ".join([w for w in str(x).split() if w not in PAROLE_DA_ELIMINARE])
)

# 3. NUOVO TF-IDF (Partiamo da zero)
tfidf_final = TfidfVectorizer(max_features=100)
X_tfidf_final = tfidf_final.fit_transform(df_es['clean_text_final'])
feat_names = tfidf_final.get_feature_names_out()

# 4. CALCOLO CORRELAZIONI
y_scores = df_es['score'].fillna(df_es['score'].mean()).values
corrs_final = [np.corrcoef(X_tfidf_final[:,i].toarray().flatten(), y_scores)[0,1] for i in range(X_tfidf_final.shape[1])]

# 5. CREAZIONE E SALVATAGGIO DEL GRAFICO (Tutto il codice, senza omissioni)
sorted_indices = np.argsort(corrs_final)
top_neg_idx = sorted_indices[:15]
top_pos_idx = sorted_indices[-15:]

plt.figure(figsize=(12, 8))
# Disegna le barre rosse (correlazioni negative)
plt.barh([feat_names[i] for i in top_neg_idx], [corrs_final[i] for i in top_neg_idx], color='red')
# Disegna le barre verdi (correlazioni positive)
plt.barh([feat_names[i] for i in top_pos_idx], [corrs_final[i] for i in top_pos_idx], color='green')

plt.title('Parole più correlate con Score Alto (Verde) e Basso (Rosso) - FINALE', fontweight='bold')
plt.axvline(0, color='black', linewidth=0.8)

# Salvataggio
OUTPUT_DIR = "/Users/alessiadelconte/codici/progetto NPL/output_eda_completa"
plt.savefig(os.path.join(OUTPUT_DIR, '08_TFIDF_PERFETTO.png'), bbox_inches='tight')
plt.close()

print(f"Grafico salvato! Controlla la cartella. 'great' è nel vocabolario? {'great' in feat_names}")

KeyError: 'clean_text'